In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

chrome_options = Options()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
actions = ActionChains(driver)

yetenek_sozlugu = {
    "Python": ["python"],
    "SQL": ["sql", "veritabanı", "database"],
    "Excel": ["excel", "ms office"],
    "İngilizce": ["ingilizce", "english"],
    "İletişim": ["iletişim", "diksiyon", "ikna"],
    "Liderlik": ["liderlik", "yönetim", "ekip yönetimi"],
    "Analiz": ["analiz", "reporting", "raporlama"],
    "Takım_Calısması": ["takım çalışması", "ekip çalışması"]
}

def veri_topla_ve_analiz_et(sayfa_sayisi=1):
    sonuclar = []

    for sayfa in range(1, sayfa_sayisi + 1):
        url = f"https://www.yenibiris.com/is-ilanlari?sayfa={sayfa}"
        driver.get(url)
        time.sleep(5) 

        ilanlar = driver.find_elements(By.CLASS_NAME, "listViewRows")

        for ilan in ilanlar:
            try:
                baslik_alani = ilan.find_element(By.CLASS_NAME, "gtmTitle")
                actions.move_to_element(baslik_alani).perform()
                time.sleep(2) 
                detay_metni = ""
                try:
                    detay_alani = ilan.find_element(By.CLASS_NAME, "detailContent")
                    detay_metni = detay_alani.text.lower()
                except:
                    continue 

                bulunan_yetenekler = {}
                for yetenek_adi, anahtar_kelimeler in yetenek_sozlugu.items():
                    if any(kelime in detay_metni for kelime in anahtar_kelimeler):
                        bulunan_yetenekler[yetenek_adi] = 1
                    else:
                        bulunan_yetenekler[yetenek_adi] = 0
                
                bulunan_yetenekler["Pozisyon"] = baslik_alani.text
                sonuclar.append(bulunan_yetenekler)
                print(f"İşlendi: {baslik_alani.text}")

            except Exception as e:
                print(f"Hata: {e}")
                continue

    return pd.DataFrame(sonuclar)

print("Yenibiriş ilanları analiz ediliyor...")
df = veri_topla_ve_analiz_et(sayfa_sayisi=101) 
driver.quit()

print("\n--- Analiz Tablosu (İlk 5 Satır) ---")
print(df.head())

df.to_csv("ik_yetenek_matrisi.csv", index=False, encoding="utf-8-sig")

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor

chrome_options = Options()
chrome_options.add_argument("--headless") 
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

yetenek_sozlugu = {
    "Python": ["python"],
    "SQL": ["sql", "veritabanı", "database"],
    "Excel": ["excel", "ms office"],
    "İngilizce": ["ingilizce", "english"],
    "İletişim": ["iletişim", "ikna", "diksiyon"],
    "Liderlik": ["liderlik", "yönetim", "management"],
    "Analiz": ["analiz", "raporlama", "reporting"],
    "Takım_Calısması": ["takım çalışması", "ekip çalışması"],
    "Agile": ["agile", "scrum"]
}

def detay_analiz_et(link):
    """Her ilanın içine tarayıcı olmadan girip hızlıca yetenekleri sayar."""
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        res = requests.get(link, headers=headers, timeout=5)
        soup = BeautifulSoup(res.content, 'html.parser')
        
        text = soup.get_text().lower()

        bulunanlar = {y: (1 if any(k in text for k in kv) else 0) for y, kv in yetenek_sozlugu.items()}
        
        baslik = soup.find("h1")
        bulunanlar["Pozisyon"] = baslik.text.strip() if baslik else "Bilinmiyor"
        bulunanlar["Link"] = link
        bulunanlar["Beceri_Sayisi"] = sum(1 for y in yetenek_sozlugu if bulunanlar[y] == 1)
        
        return bulunanlar
    except:
        return None

def topla_ve_bitir(sayfa_sayisi=70): 
    ilan_linkleri = []
    
    print(f"{sayfa_sayisi} sayfa taranıyor, linkler toplanıyor...")
    for sayfa in range(1, sayfa_sayisi + 1):
        url = f"https://www.eleman.net/is-ilanlari?sy={sayfa}"
        driver.get(url)
        time.sleep(1)
        
        linkler = driver.find_elements(By.CSS_SELECTOR, ".ilan_listeleme_bol > a")
        for link in linkler:
            ilan_linkleri.append(link.get_attribute("href"))
        
        if len(ilan_linkleri) >= 1000: break
        print(f"Toplanan link: {len(ilan_linkleri)}")

    driver.quit()

    print(f"\n{len(ilan_linkleri)} ilan paralel olarak işleniyor... (Çok hızlı)")
    with ThreadPoolExecutor(max_workers=20) as executor:
        sonuclar = list(executor.map(detay_analiz_et, ilan_linkleri))

    df = pd.DataFrame([s for s in sonuclar if s is not None])
    df.to_csv("is_ilanlari_1000_analiz.csv", index=False, encoding="utf-8-sig")
    print(f"\nBİTTİ! 1000+ veri 'is_ilanlari_1000_analiz.csv' olarak hazır.")
    return df

df_final = topla_ve_bitir(75)